# Lesson 12 · AI Agents III · Multi-agent · Demo
## Um problema, três arquiteturas, medidas

**DCA0305 · Machine Learning-Based Systems Design · UFRN**

Nas aulas 10 e 11, construímos **um** agente: um modelo, um loop e três ferramentas que agora vêm de um servidor MCP. Hoje dividimos o trabalho entre vários agentes e, em vez de discutir qual arquitetura é melhor, **medimos**.

| Parte | O que fazemos | Agentes |
|---|---|---|
| **1 · O loop vira um grafo** | o agente da aula 11, reescrito em LangGraph | 1 |
| **2 · Gerador + verificador** | um segundo agente confere a resposta contra o estoque | 2 |
| **3 · Supervisor + especialistas** | um agente distribui o trabalho entre dois especialistas | 3 + redator |
| **4 · Medindo as três** | 8 perguntas com resposta conhecida: acerto, chamadas ao modelo e tempo | — |
| **5 · As mesmas três no CrewAI** | as três arquiteturas em outro framework, medidas com as mesmas 8 perguntas | 1 a 3 + gerente |

As ferramentas são as mesmas da aula 11, servidas pelo **mesmo `server.py`**, sem mudar uma linha.

**Como rodar.** No Colab, escolha o runtime **T4 GPU** (Ambiente de execução → Alterar o tipo de ambiente de execução) e rode tudo de cima a baixo. O notebook inteiro leva uns 40 minutos, quase todos nas Partes 4 e 5, que rodam 48 perguntas.

---
## 🧰 Preparação

A célula abaixo confere cada passo e só faz o que falta. Pode rodar quantas vezes quiser. Na primeira vez no Colab, leva uns 5 minutos. Se ela pedir para **reiniciar a sessão**, é porque o `pip` atualizou uma biblioteca que já estava carregada: reinicie e rode a célula de novo.

Uma novidade em relação à aula 11: **o servidor e o notebook usam versões diferentes da biblioteca `mcp`**. O FastMCP 4 do servidor fala a especificação de 2026 e precisa do `mcp` 2.x. O LangGraph e o CrewAI ainda usam o `mcp` 1.x. As duas versões não cabem no mesmo ambiente Python, então instalamos as bibliotecas do servidor numa pasta só dele, `servidor_libs/`. Isso só funciona porque **o servidor é outro programa**, que conversa com o notebook por mensagens, como vimos na aula 11.

In [1]:
# @title 🧰 Preparação: confere as bibliotecas, o estoque, o Ollama e o modelo, e só instala o que faltar
import os, shutil, subprocess, sys, time, requests

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3:8b"
IN_COLAB = "google.colab" in sys.modules
sh = get_ipython().system                      # roda um comando de shell mostrando a saída na célula
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]  # o pip do próprio notebook
NOTEBOOK_LIBS = "langgraph==1.2.20 langchain-ollama==1.1.0 langchain-mcp-adapters==0.3.2 crewai==1.15.36 crewai-tools[mcp]==1.15.36"
SERVER_LIBS = os.path.abspath("servidor_libs")
RESTART = ("o pip atualizou bibliotecas que esta sessão já tinha carregado. Reinicie a sessão "
           "(no Colab, Ambiente de execução → Reiniciar sessão; no Jupyter, Kernel → Restart) "
           "e rode esta célula de novo.")

# 1 · As bibliotecas do notebook (LangGraph, adaptador MCP e CrewAI) estão instaladas?
try:
    import langgraph, langchain_mcp_adapters, crewai, crewai_tools
    print("✅ LangGraph e CrewAI já instalados")
except ImportError:
    print("⏳ instalando LangGraph, o adaptador MCP e o CrewAI (cerca de 2 minutos)...")
    sh(f"pip -q install --disable-pip-version-check {NOTEBOOK_LIBS}")
    print("✅ LangGraph e CrewAI instalados")
try:
    from langgraph.graph import StateGraph
    from langchain_mcp_adapters.client import MultiServerMCPClient
    from crewai import Agent
    from crewai_tools import MCPServerAdapter
except ImportError as e:
    raise RuntimeError(RESTART) from e

# 2 · O servidor tem as bibliotecas dele, numa pasta separada?
if os.path.exists(os.path.join(SERVER_LIBS, "fastmcp")):
    print("✅ bibliotecas do servidor já estão em servidor_libs/")
else:
    print("⏳ instalando o FastMCP 4.1.4 só para o servidor, em servidor_libs/ ...")
    sh(f"pip -q install --disable-pip-version-check --target {SERVER_LIBS} fastmcp==4.1.4 > /dev/null 2>&1")
    if not os.path.exists(os.path.join(SERVER_LIBS, "fastmcp")):
        raise RuntimeError("a instalação do FastMCP falhou. Rode a célula de novo.")
    print("✅ bibliotecas do servidor instaladas em servidor_libs/")

# 3 · O estoque da aula 10 está ao lado do notebook?
CSV = "cars.csv"
if os.path.exists(CSV):
    print(f"✅ {CSV} já está aqui")
else:
    url = "https://raw.githubusercontent.com/ivanovitchm/aiengineering/main/lessons/week07/cars.csv"
    r = requests.get(url)
    r.raise_for_status()
    open(CSV, "wb").write(r.content)
    print(f"✅ {CSV} baixado")

def server_is_up() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

# 4 · O programa ollama existe nesta máquina?
if shutil.which("ollama"):
    print("✅ Ollama já instalado")
elif IN_COLAB:
    print("⏳ instalando o Ollama (cerca de 1 minuto)...")
    sh("apt-get install -y -qq zstd > /dev/null 2>&1 || "
       "(apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)")
    sh("curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2")
    if not shutil.which("ollama"):
        raise RuntimeError("a instalação falhou. Rode a célula de novo.")
    print("✅ Ollama instalado")
else:
    raise RuntimeError("Ollama não encontrado. Instale em https://ollama.com/download e rode esta célula de novo.")

# 5 · O servidor do Ollama está respondendo?
if server_is_up():
    print("✅ servidor do Ollama já está de pé")
else:
    print("⏳ subindo o servidor do Ollama...")
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)  # continua vivo depois que a célula termina
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)
    if not server_is_up():
        raise RuntimeError("o servidor não respondeu. Veja o arquivo ollama.log")
    print("✅ servidor do Ollama de pé")

# 6 · O modelo já foi baixado?
installed = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
if MODEL in installed:
    print(f"✅ {MODEL} já baixado")
else:
    print(f"⏳ baixando {MODEL} (~5 GB). Se a conexão cair, rode a célula de novo: o download continua.")
    sh(f"ollama pull {MODEL}")
    print(f"✅ {MODEL} baixado")

version = requests.get(f"{OLLAMA_URL}/api/version").json()["version"]
print(f"\n🚀 pronto · LangGraph {langgraph.__version__ if hasattr(langgraph, '__version__') else '1.2'} · "
      f"CrewAI {crewai.__version__} · Ollama {version} · modelo {MODEL} · {'Colab' if IN_COLAB else 'máquina local'}")

✅ LangGraph e CrewAI já instalados


✅ bibliotecas do servidor já estão em servidor_libs/
✅ cars.csv já está aqui
✅ Ollama já instalado
✅ servidor do Ollama já está de pé
✅ qwen3:8b já baixado

🚀 pronto · LangGraph 1.2 · CrewAI 1.15.36 · Ollama 0.40.11 · modelo qwen3:8b · máquina local


In [2]:
# @title 📡 O server.py da aula 11, sem mudar uma linha
SERVER_PY = r'''
from typing import Literal
import pandas as pd
from fastmcp import FastMCP                                   # ←

cars = pd.read_csv("cars.csv")
mcp = FastMCP("estoque-natal")                                # ← o servidor, com um nome

# ─── TOOLS: o modelo decide quando chamar ───

@mcp.tool                                                     # ← isto substitui o JSON Schema escrito à mão
def search_cars(max_price: float, body: Literal["SUV", "sedan", "hatch"] | None = None) -> dict:
    """Search the used-car stock by budget and body type.
    Returns how many cars matched and the 5 newest ones (car_id, model, year, km, price)."""
    r = cars[cars.price_brl <= max_price]
    if body is not None:
        r = r[r.body == body]
    r = r.sort_values(["year", "km"], ascending=[False, True])
    cols = ["car_id", "brand", "model", "year", "km", "price_brl", "transmission"]
    return {"total_found": len(r), "cars": r[cols].head(5).to_dict("records")}

@mcp.tool                                                     # ←
def car_details(car_id: str) -> dict:
    """Get all the details of one car: trunk size (liters), fuel consumption (km/l),
    color and city. Use it with a car_id returned by search_cars."""
    r = cars[cars.car_id == car_id]
    if r.empty:
        return {"error": f"no car with id {car_id}"}
    return r.to_dict("records")[0]

@mcp.tool                                                     # ←
def compare_cars(car_ids: list[str]) -> dict:
    """Compare 2 to 4 cars side by side (year, km, price, transmission, trunk size, consumption).
    Use it when the user wants to choose between specific cars."""
    r = cars[cars.car_id.isin(car_ids)]
    if r.empty:
        return {"error": f"none of these ids exist: {car_ids}"}
    cols = ["car_id", "brand", "model", "year", "km", "price_brl", "transmission", "trunk_l", "consumption_kml"]
    def winners(col, highest):
        target = r[col].max() if highest else r[col].min()
        return list(r[r[col] == target].car_id)              # mais de um id = empate
    best = {"largest_trunk": winners("trunk_l", True),
            "most_economical": winners("consumption_kml", True),   # mais km por litro = consome menos
            "cheapest": winners("price_brl", False),
            "newest": winners("year", True),
            "lowest_km": winners("km", False)}
    return {"cars": r[cols].to_dict("records"), "best": best,
            "missing": [i for i in car_ids if i not in set(r.car_id)]}


# ─── RESOURCE: a aplicação lê, por um endereço ───

@mcp.resource("estoque://schema")                             # ←
def schema() -> str:
    """The columns of the stock table."""
    return ", ".join(cars.columns)


# ─── PROMPT: o usuário escolhe usar ───

@mcp.prompt                                                   # ←
def recomendar(orcamento: int, carroceria: str) -> str:
    """Ask for a recommendation with budget and body type filled in."""
    return f"Quero um {carroceria} até R$ {orcamento}. Busque no estoque e recomende 3 carros, citando o car_id."


if __name__ == "__main__":
    mcp.run()                                                 # ← fala MCP pelo stdio (entrada e saída padrão)
'''
open("server.py", "w").write(SERVER_PY.lstrip("\n"))
print("server.py gravado: o mesmo da aula 11 (search_cars, car_details, compare_cars)")

server.py gravado: o mesmo da aula 11 (search_cars, car_details, compare_cars)


In [3]:
# @title 🔧 Utilitários: como subir o servidor, o modelo e funções para imprimir
import json, re
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama

# Como o notebook sobe o servidor: o mesmo Python, mas com as bibliotecas de servidor_libs/ na frente
SERVIDOR = {"estoque": {"transport": "stdio", "command": sys.executable, "args": ["server.py"],
                        "env": {"PYTHONPATH": SERVER_LIBS,
                                "FASTMCP_LOG_LEVEL": "ERROR", "FASTMCP_SHOW_SERVER_BANNER": "false"}}}   # sem log na tela

CHAMADAS = 0                                   # quantas vezes o modelo foi chamado na pergunta atual

def modelo(pensar=False, max_tokens=600):
    """O qwen3:8b no Ollama, com temperatura zero. pensar=True liga o raciocínio do qwen3 (mais lento)."""
    return ChatOllama(model=MODEL, temperature=0, num_ctx=8192, num_predict=max_tokens, reasoning=pensar)

async def chamar(llm, mensagens):
    """Chama o modelo e conta a chamada."""
    global CHAMADAS
    CHAMADAS += 1
    return await llm.ainvoke(mensagens)

def texto(conteudo):
    """O texto de um resultado de ferramenta (o adaptador MCP entrega uma lista de blocos)."""
    if isinstance(conteudo, str):
        return conteudo
    return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in conteudo)

def resumo(resultado):
    """Uma linha sobre o resultado de uma ferramenta, para o trace."""
    try:
        d = json.loads(resultado)
    except json.JSONDecodeError:
        return "erro: " + resultado.splitlines()[0][:90]
    if "error" in d:
        return "erro: " + d["error"]
    if "total_found" in d:
        return f'{d["total_found"]} encontrados; devolveu {[c["car_id"] for c in d["cars"]]}'
    if "best" in d:
        return "melhor em cada critério: " + ", ".join(f"{k} {'/'.join(v)}" for k, v in d["best"].items())
    return f'{d["car_id"]} {d["model"]} {d["body"]}: {d["consumption_kml"]} km/l, porta-malas {d["trunk_l"]} L, {d["color"]}, {d["city"]}'

def mostrar(r):
    """O trace de uma execução: cada pedido de ferramenta, cada resultado e as mensagens entre agentes."""
    for m in r["messages"][1:]:
        nome = getattr(m, "name", None)
        if isinstance(m, AIMessage) and m.tool_calls:
            for c in m.tool_calls:
                print(f"   🤖 pede        → {c['name']}({json.dumps(c['args'], ensure_ascii=False)})")
        elif isinstance(m, ToolMessage):
            print(f"   📡 servidor    → {resumo(texto(m.content))}")
        elif nome == "verificador":
            print(f"   🔍 verificador → {m.content[:300]}")
        elif nome == "supervisor":
            print(f"   🧭 supervisor  → {m.content[:160]}")
        elif nome in ("buscador", "comparador"):
            print(f"   📋 {nome:<11} → {m.content.replace(chr(10), ' ')[:160]}…")
    print(f"\n💬 {r['resposta']}\n")
    print(f"⏱ {r['chamadas']} chamadas ao modelo · {r['segundos']} s")

async def rodar(app, pergunta, ver=True):
    """Roda uma arquitetura numa pergunta e devolve a resposta, a trajetória, as chamadas e o tempo."""
    global CHAMADAS
    CHAMADAS, t = 0, time.time()
    out = await app.ainvoke({"messages": [HumanMessage(pergunta)]}, {"recursion_limit": 40})
    msgs = out["messages"]
    r = {"resposta": next(m.content for m in reversed(msgs)
                          if isinstance(m, AIMessage) and not m.tool_calls and m.name is None),
         "ferramentas": [c["name"] for m in msgs if isinstance(m, AIMessage) for c in m.tool_calls],
         "chamadas": CHAMADAS, "segundos": round(time.time() - t, 1), "messages": msgs}
    if ver:
        print(f"👤 {pergunta}\n")
        mostrar(r)
    return r

print("utilitários prontos")

utilitários prontos


---
# Parte 1 · O loop vira um grafo

O agente da aula 10 era um `while` com três passos. No LangGraph, o mesmo loop vira um **grafo**: cada passo é um **nó**, e as setas dizem para onde ir depois.

| Aula 10 (`while`) | LangGraph |
|---|---|
| `messages`, a lista da conversa | o **estado**, que passa de nó em nó |
| chamar o modelo (REASON) | o nó `modelo` |
| executar a ferramenta (ACT) e devolver o resultado (OBSERVE) | o nó `ferramentas`, que volta para o `modelo` |
| `if not msg.get("tool_calls"): return` | uma **aresta condicional**: pediu ferramenta? vai para `ferramentas`. Senão, fim |

```
          ┌──────────── pediu ferramenta ────────────┐
          │                                          ▼
START ─► modelo                                 ferramentas
          ▲                                          │
          └──────────────── resultado ───────────────┘
          │
          └─► respondeu com texto ─► END
```

Primeiro, as ferramentas. O `MultiServerMCPClient` sobe o `server.py` e pergunta a lista, com o mesmo `tools/list` da aula 11.

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

TOOLS = await MultiServerMCPClient(SERVIDOR).get_tools()     # sobe o server.py e faz tools/list
for t in TOOLS:
    print(f"🔧 {t.name:<13} {t.description.splitlines()[0]}")

🔧 search_cars   Search the used-car stock by budget and body type.
🔧 car_details   Get all the details of one car: trunk size (liters), fuel consumption (km/l),
🔧 compare_cars  Compare 2 to 4 cars side by side (year, km, price, transmission, trunk size, consumption).


Agora o grafo. Ele é uma função porque, nas Partes 2 e 3, vamos montar outros agentes com ela, cada um com as suas ferramentas e as suas instruções.

In [5]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

SYSTEM = """You are a used-car buying assistant in Natal, Brazil. Answer in the user's language.
Use the tools to look up real listings; never invent a car.
Always cite the car_id of every car you mention."""          # o mesmo system prompt da aula 11

class Estado(TypedDict):
    messages: Annotated[list, add_messages]                   # a conversa; cada nó acrescenta mensagens

def agente(ferramentas, sistema=SYSTEM):
    """O agente da aula 10 como grafo: modelo → ferramentas → modelo ... até responder com texto."""
    llm = modelo().bind_tools(ferramentas)                    # o modelo recebe as descrições das ferramentas
    async def no_modelo(estado):                              # REASON
        return {"messages": [await chamar(llm, [SystemMessage(sistema)] + estado["messages"])]}
    g = StateGraph(Estado)
    g.add_node("modelo", no_modelo)
    g.add_node("ferramentas", ToolNode(ferramentas, handle_tool_errors=True))   # ACT: chama o servidor MCP
    g.add_edge(START, "modelo")
    g.add_conditional_edges("modelo", tools_condition, {"tools": "ferramentas", END: END})
    g.add_edge("ferramentas", "modelo")                       # OBSERVE: o resultado volta ao modelo
    return g.compile()

unico = agente(TOOLS)
PEDIDO = "Quero um SUV para a família, até uns 80 mil. O que você recomenda?"
r = await rodar(unico, PEDIDO)

👤 Quero um SUV para a família, até uns 80 mil. O que você recomenda?

   🤖 pede        → search_cars({"max_price": 80000, "body": "SUV"})
   📡 servidor    → 22 encontrados; devolveu ['c082', 'c081', 'c109', 'c061', 'c003']

💬 Aqui estão algumas opções de SUVs que você pode considerar dentro do seu orçamento de até R$80 mil:

1. **Renault Duster 2021 (Automático)** - R$75.000, com 56.500 km.  
   - Car_id: `c082`

2. **Renault Duster 2021 (Manual)** - R$71.000, com 67.700 km.  
   - Car_id: `c081`

3. **Volkswagen Nivus 2021 (Manual)** - R$75.500, com 70.300 km.  
   - Car_id: `c109`

4. **Jeep Renegade 2020 (Manual)** - R$76.500, com 82.700 km.  
   - Car_id: `c061`

5. **Chevrolet Tracker 2019 (Automático)** - R$74.500, com 91.000 km.  
   - Car_id: `c003`

Se quiser, posso mostrar mais detalhes sobre algum desses carros ou comparar alguns deles. Qual você gostaria de ver mais informações?

⏱ 2 chamadas ao modelo · 25.1 s


### 🔍 O que aconteceu

- O modelo pediu **uma** ferramenta, `search_cars` com `max_price: 80000` e `body: "SUV"`. O servidor achou 22 SUVs e devolveu os 5 mais novos: c082, c081, c109, c061 e c003.
- A resposta lista os cinco com preço, quilometragem e `car_id`, e termina oferecendo detalhes ou uma comparação.
- Foram **2 chamadas ao modelo**: uma para pedir a ferramenta e outra para escrever. No grafo, o caminho foi `modelo → ferramentas → modelo → fim`.

É o mesmo agente da aula 11. Mudou só a forma: o `while` virou a aresta que volta das ferramentas para o modelo, e o `if` virou a aresta condicional. Esse agente é a linha de base das próximas partes.

---
# Parte 2 · Gerador + verificador

O agente da Parte 1 continua sendo o **gerador**. Depois que ele responde, entra um **verificador**, que confere cada afirmação contra o estoque. Se achar um erro, devolve a resposta ao gerador com o motivo.

```
START ─► gerador ⇄ ferramentas
            │
            ▼ respondeu
        verificador ── ok ─► END
            │
            └── achou erro ─► gerador (no máximo 2 vezes)
```

Três decisões de projeto, cada uma com um porquê:

1. **A consulta é código, o julgamento é do modelo.** Um regex acha os `car_id` da resposta e o código chama `car_details` para cada um. Não há por que gastar uma chamada ao modelo para isso.
2. **O verificador pensa antes de julgar.** Ele usa o modo de raciocínio do qwen3 (`pensar=True`), mais lento, mas mais atento a rankings como "o mais econômico".
3. **Toda objeção precisa de evidência.** O verificador tem de citar o resultado de ferramenta que contradiz a resposta. Se um `car_id` ou um número da citação não aparece nos resultados, o código descarta a objeção. Um verificador também alucina, e essa regra impede que ele "corrija" o que estava certo.

In [6]:
from pydantic import BaseModel, Field

class Problema(BaseModel):
    afirmacao: str = Field(description="the wrong claim, copied from the answer")
    evidencia: str = Field(description="the tool result that contradicts the claim, quoted with its car_id and numbers")

class Revisao(BaseModel):
    problemas: list[Problema] = Field(description="wrong claims; empty list if the answer is correct")

VERIFICADOR = """You are a strict fact checker for a used-car assistant. You get the user's request,
the tool calls and results, and the assistant's answer. The tool results are the only source of truth.
Report a problem only when a tool result contradicts a claim in the answer
(a number, a car_id, a body type such as SUV, a ranking such as cheapest, newest or lowest km).
Check every ranking with the numbers: more km/l means MORE economical; fewer km is better; a lower price is cheaper.
When compare_cars was called, its "best" field is the ground truth for rankings.
If the answer is correct, return an empty list."""

def trajetoria(msgs):
    """As chamadas de ferramenta e os resultados, em texto, para o verificador ler."""
    linhas = []
    for m in msgs:
        if isinstance(m, AIMessage):
            linhas += [f"CALL {c['name']}({json.dumps(c['args'], ensure_ascii=False)})" for c in m.tool_calls]
        elif isinstance(m, ToolMessage):
            linhas.append(f"RESULT {texto(m.content)}")
    return "\n".join(linhas)

def tem_evidencia(evidencia, resultados):
    """Cada car_id e cada número da evidência precisa aparecer nos resultados das ferramentas."""
    fatos = re.findall(r"c\d{3}|\d+(?:\.\d+)?", evidencia)
    return bool(fatos) and all(f in resultados.replace(" ", "") for f in fatos)

def gerador_verificador(ferramentas, max_revisoes=2):
    detalhes = {t.name: t for t in ferramentas}["car_details"]
    gerador = modelo().bind_tools(ferramentas)
    juiz = modelo(pensar=True, max_tokens=4000).with_structured_output(Revisao)   # o verificador pensa antes de julgar

    async def no_gerador(estado):
        return {"messages": [await chamar(gerador, [SystemMessage(SYSTEM)] + estado["messages"])]}

    async def no_verificador(estado):
        msgs = estado["messages"]
        resposta = msgs[-1].content
        ids = sorted(set(re.findall(r"\bc\d{3}\b", resposta)))            # código: quais carros a resposta cita?
        consultas = [f"CALL car_details({i})\nRESULT {texto(await detalhes.ainvoke({'car_id': i}))}" for i in ids]
        resultados = trajetoria(msgs) + "\n" + "\n".join(consultas)
        rev = await chamar(juiz, [SystemMessage(VERIFICADOR), HumanMessage(
            f"USER REQUEST:\n{msgs[0].content}\n\nTOOL CALLS AND RESULTS:\n{resultados}\n\nASSISTANT ANSWER:\n{resposta}")])
        validos = [p for p in rev.problemas if tem_evidencia(p.evidencia, resultados)]
        descartados = len(rev.problemas) - len(validos)
        if not validos:                                                    # aprovado
            return {"messages": [AIMessage(f"consultou {', '.join(ids) or 'nada'}; ok"
                                           + (f" ({descartados} objeção sem evidência descartada)" if descartados else ""),
                                           name="verificador")]}
        retorno = "A reviewer found these problems. Fix them, calling tools again if needed, and rewrite the answer:\n"
        retorno += "\n".join(f'- "{p.afirmacao}" is contradicted by: {p.evidencia}' for p in validos)
        return {"messages": [HumanMessage(retorno, name="verificador")]}  # volta ao gerador

    def depois_do_gerador(estado):
        return "ferramentas" if estado["messages"][-1].tool_calls else "verificador"

    def depois_do_verificador(estado):
        revisoes = sum(1 for m in estado["messages"] if isinstance(m, HumanMessage) and m.name == "verificador")
        aprovado = isinstance(estado["messages"][-1], AIMessage)
        return END if aprovado or revisoes > max_revisoes else "gerador"

    g = StateGraph(Estado)
    g.add_node("gerador", no_gerador)
    g.add_node("ferramentas", ToolNode(ferramentas, handle_tool_errors=True))
    g.add_node("verificador", no_verificador)
    g.add_edge(START, "gerador")
    g.add_conditional_edges("gerador", depois_do_gerador, ["ferramentas", "verificador"])
    g.add_edge("ferramentas", "gerador")
    g.add_conditional_edges("verificador", depois_do_verificador, ["gerador", END])
    return g.compile()

dupla = gerador_verificador(TOOLS)
print("gerador + verificador pronto")

gerador + verificador pronto


🔮 **Antes de rodar.** Uma pergunta simples: entre dois Dusters, qual é o mais econômico? O c081 faz 10,9 km/l e o c082 faz 10,4 km/l. Primeiro o agente único da Parte 1. Ele acerta?

In [7]:
ECONOMICO = "Entre o c081 e o c082, qual é o mais econômico?"
r = await rodar(unico, ECONOMICO)

👤 Entre o c081 e o c082, qual é o mais econômico?

   🤖 pede        → car_details({"car_id": "c081"})
   🤖 pede        → car_details({"car_id": "c082"})
   📡 servidor    → c081 Duster SUV: 10.9 km/l, porta-malas 475 L, branco, Natal
   📡 servidor    → c082 Duster SUV: 10.4 km/l, porta-malas 475 L, cinza, Macaíba

💬 O carro c082 é o mais econômico, com um consumo de 10,4 km/l, em comparação ao c081, que tem um consumo de 10,9 km/l.

⏱ 2 chamadas ao modelo · 8.5 s


O agente chamou a `car_details` duas vezes e recebeu os números certos: c081 com 10,9 km/l e c082 com 10,4 km/l. E escreveu: **"O carro c082 é o mais econômico"**, citando os dois números corretos logo em seguida.

É o mesmo tipo de erro do slide 5. Os números vieram da ferramenta; a frase veio do modelo, e ninguém conferiu a frase. Mais km/l é mais econômico, então o certo é o c081.

Agora a mesma pergunta para o gerador + verificador.

In [8]:
r = await rodar(dupla, ECONOMICO)

👤 Entre o c081 e o c082, qual é o mais econômico?

   🤖 pede        → car_details({"car_id": "c081"})
   🤖 pede        → car_details({"car_id": "c082"})
   📡 servidor    → c081 Duster SUV: 10.9 km/l, porta-malas 475 L, branco, Natal
   📡 servidor    → c082 Duster SUV: 10.4 km/l, porta-malas 475 L, cinza, Macaíba
   🔍 verificador → A reviewer found these problems. Fix them, calling tools again if needed, and rewrite the answer:
- "O carro c082 é o mais econômico" is contradicted by: consumption_kml: 10.4 vs 10.9 (c081 tem maior consumo_kml, ou seja, é mais econômico)
   🔍 verificador → consultou c081, c082; ok

💬 O carro c081 é o mais econômico, com um consumo de 10,9 km/l, em comparação ao c082, que tem um consumo de 10,4 km/l.

⏱ 5 chamadas ao modelo · 47.1 s


### 🔍 O que aconteceu

- O gerador escreveu a mesma resposta errada. Aí o **código** procurou os `car_id` citados (c081 e c082), consultou a `car_details` de cada um e entregou os dados ao verificador.
- O verificador apontou o problema **com evidência**: a frase "O carro c082 é o mais econômico" é contradita por `consumption_kml: 10.4 vs 10.9`. A objeção passou pela checagem `tem_evidencia` e voltou ao gerador.
- O gerador reescreveu: **"O carro c081 é o mais econômico"**. Na segunda volta, o verificador consultou de novo e aprovou (`ok`).
- O preço: **5 chamadas e 47 s**, contra 2 chamadas e 9 s do agente único. Uma devolução custou cinco vezes o tempo.

Repare em quem fez o quê. Consultar os dados foi trabalho do código, que não erra a consulta. O modelo só julgou se a frase batia com os dados, e só pôde objetar mostrando de onde tirou a objeção.

---
# Parte 3 · Supervisor + especialistas

Agora dividimos **por especialidade**. Um **supervisor** lê o pedido e decide quem age: o **buscador**, que só tem `search_cars` e `car_details`, ou o **comparador**, que só tem `compare_cars`. Cada especialista é o agente da Parte 1, com menos ferramentas e outra instrução. No fim, um **redator** escreve a resposta com base nos relatórios.

```
START ─► supervisor ─► buscador   ─┐
            ▲      └─► comparador ─┤
            └──────── relatório ◄──┘
            │
            └─► "answer" ─► redator ─► END
```

Repare num detalhe: **o especialista não vê a conversa**. Ele recebe só a tarefa que o supervisor escreveu, e o supervisor recebe só o relatório final dele. É assim que o contexto fica isolado, e é também onde a informação se perde: tudo o que o supervisor não escreveu na tarefa, o especialista não sabe.

In [9]:
from typing import Literal

class Rota(BaseModel):
    proximo: Literal["buscador", "comparador", "answer"] = Field(description="who acts next")
    tarefa: str = Field(description="the instruction for the specialist, with car_ids and filters; empty for 'answer'")

SUPERVISOR = """You coordinate two specialists for a used-car assistant in Natal, Brazil.
- buscador: finds cars by budget and body type, and gets the details of one car (tools: search_cars, car_details).
- comparador: compares 2 to 4 specific car_ids side by side (tool: compare_cars).
Read the request and the reports so far, and pick who acts next with a precise task.
Pick 'answer' as soon as the reports are enough to answer the user."""

REDATOR = SYSTEM + "\nWrite the final answer to the user using only the specialists' reports."

class EstadoSupervisor(TypedDict):
    messages: Annotated[list, add_messages]
    relatorios: list                                          # o que os especialistas devolveram
    proximo: str
    tarefa: str

def supervisor_especialistas(ferramentas, max_passos=4):
    t = {f.name: f for f in ferramentas}
    especialistas = {
        "buscador":   agente([t["search_cars"], t["car_details"]], SYSTEM + "\nYou are the search specialist. Do the task and report the facts."),
        "comparador": agente([t["compare_cars"]], SYSTEM + "\nYou are the comparison specialist. Do the task and report the facts."),
    }
    roteador = modelo(max_tokens=200).with_structured_output(Rota)
    redator = modelo(max_tokens=900)

    def briefing(estado):                                     # o que o supervisor e o redator leem
        rel = "\n".join(estado.get("relatorios", [])) or "(none yet)"
        return f"USER REQUEST:\n{estado['messages'][0].content}\n\nSPECIALIST REPORTS:\n{rel}"

    async def no_supervisor(estado):
        if len(estado.get("relatorios", [])) >= max_passos:  # teto de passos
            return {"proximo": "answer", "tarefa": ""}
        rota = await chamar(roteador, [SystemMessage(SUPERVISOR), HumanMessage(briefing(estado))])
        return {"proximo": rota.proximo, "tarefa": rota.tarefa,
                "messages": [AIMessage(f"{rota.proximo}: {rota.tarefa}", name="supervisor")]}

    def especialista(nome):
        async def no(estado):
            out = await especialistas[nome].ainvoke({"messages": [HumanMessage(estado["tarefa"])]})   # só a tarefa
            relatorio = f"[{nome}] {out['messages'][-1].content}"
            return {"messages": out["messages"][1:-1] + [AIMessage(relatorio, name=nome)],
                    "relatorios": estado.get("relatorios", []) + [relatorio]}
        return no

    async def no_redator(estado):
        return {"messages": [await chamar(redator, [SystemMessage(REDATOR), HumanMessage(briefing(estado))])]}

    g = StateGraph(EstadoSupervisor)
    g.add_node("supervisor", no_supervisor)
    for nome in especialistas:
        g.add_node(nome, especialista(nome))
        g.add_edge(nome, "supervisor")
    g.add_node("redator", no_redator)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", lambda e: e["proximo"],
                            {"buscador": "buscador", "comparador": "comparador", "answer": "redator"})
    g.add_edge("redator", END)
    return g.compile()

equipe = supervisor_especialistas(TOOLS)
r = await rodar(equipe, PEDIDO)

👤 Quero um SUV para a família, até uns 80 mil. O que você recomenda?

   🧭 supervisor  → buscador: Procure SUVs com preço até 80 mil reais e retorne os detalhes de 3 carros para comparação.
   🤖 pede        → search_cars({"max_price": 80000, "body": "SUV"})
   📡 servidor    → 22 encontrados; devolveu ['c082', 'c081', 'c109', 'c061', 'c003']
   📋 buscador    → [buscador] Aqui estão os detalhes de 3 carros SUVs encontrados com preço até 80 mil reais:  1. **Renault Duster 2021 (car_id: c082)**      - Ano: 2021      - Qu…
   🧭 supervisor  → comparador: Comparar os carros c081, c082 e c109 para ajudar o usuário a escolher o melhor SUV para a família dentro do orçamento de 80 mil reais.
   🤖 pede        → compare_cars({"car_ids": ["c081", "c082", "c109"]})
   📡 servidor    → melhor em cada critério: largest_trunk c081/c082, most_economical c109, cheapest c081, newest c081/c082/c109, lowest_km c082
   📋 comparador  → [comparador] Aqui está a comparação dos carros **c081**, **c082** e **c109**

### 🔍 O que aconteceu

- O supervisor mandou o **buscador** "procurar SUVs até 80 mil e retornar os detalhes de 3 carros". O buscador chamou `search_cars` e devolveu um relatório.
- Depois mandou o **comparador** comparar c081, c082 e c109. O `compare_cars` disse: maior porta-malas c081/c082, mais econômico c109, mais barato c081, menor quilometragem c082.
- Com os dois relatórios, o supervisor encerrou e o **redator** escreveu a resposta.
- A resposta é mais rica que a da Parte 1. Cada carro vem com consumo e porta-malas, e com uma frase dizendo para quem ele serve. O agente único só listou cinco carros.
- O preço: **8 chamadas e 113 s**, contra 2 chamadas e 25 s do agente único. Quatro vezes as chamadas e quatro vezes e meia o tempo.

Olhe a tarefa que o supervisor escreveu para o buscador: "os detalhes de 3 carros". O buscador não viu o pedido do usuário, só essa frase. Se o supervisor tivesse esquecido o "até 80 mil", o buscador não saberia do orçamento. Esse é o contexto que se perde na passagem.

---
# Parte 4 · Medindo as três

Um exemplo não decide nada. Para comparar arquiteturas, precisamos de um **conjunto de perguntas com resposta conhecida** e de um jeito de corrigir. São 8 perguntas: buscas com filtro, detalhes de um carro, comparações, a "pegadinha" de comparar os mais novos e um carro que não existe.

**O gabarito sai dos dados, não da nossa cabeça.** Cada resposta esperada é calculada com pandas a partir do `cars.csv`.

Corrigimos de dois jeitos:

| | Checagem automática | Juiz LLM |
|---|---|---|
| **O quê** | regras em código: o `car_id` certo aparece? algum carro fora do filtro? a ferramenta certa foi chamada? | outro modelo lê a pergunta, o gabarito e a resposta, e diz se a resposta está correta |
| **Olha** | a resposta **e a trajetória** | a resposta |
| **Vantagem** | exata, barata, repetível | entende o sentido ("o c082 é o mais econômico" está errado mesmo citando os números certos) |
| **Risco** | só pega o que a regra previu | também erra, por isso recebe o gabarito |

In [10]:
import pandas as pd
cars = pd.read_csv("cars.csv").set_index("car_id")

def ficha(cid, *cols):
    c = cars.loc[cid]
    return f"{cid} {c.brand} {c.model} " + ", ".join(f"{k} {c[k]}" for k in cols)

def busca(body, max_price):
    r = cars[(cars.body == body) & (cars.price_brl <= max_price)]
    return r.sort_values(["year", "km"], ascending=[False, True])              # a mesma ordem da search_cars

def num(x):
    return str(x).replace(".", "[,.]")                                         # 10.9 casa com 10,9 e 10.9

suvs, hatches = busca("SUV", 80000), busca("hatch", 50000)
eco = cars.loc[["c081", "c082"]].consumption_kml.idxmax()
mala = cars.loc[["c109", "c061"]].trunk_l.idxmax()
novos = suvs[suvs.year == suvs.year.max()].index

PERGUNTAS = [
    dict(q=PEDIDO, ferramenta="search_cars", deve=[suvs.index[0]], body="SUV", max_price=80000,
         gabarito=f"{len(suvs)} SUVs cost up to R$ 80,000. The 5 newest: "
                  + "; ".join(ficha(i, "year", "km", "price_brl", "transmission") for i in suvs.index[:5])
                  + ". Recommending any of them is fine."),
    dict(q="Procuro um hatch até 50 mil. Qual é o mais novo?", ferramenta="search_cars",
         deve=[hatches.index[0]], body="hatch", max_price=50000,
         gabarito=f"The newest hatch up to R$ 50,000 is {ficha(hatches.index[0], 'year', 'km', 'price_brl')}."),
    dict(q="Em que cidade está o c109 e qual é a cor dele?", ferramenta="car_details",
         deve=[cars.loc["c109"].city.lower(), cars.loc["c109"].color[:-1]],
         gabarito=ficha("c109", "city", "color")),
    dict(q="Qual é o consumo e o tamanho do porta-malas do c003?", ferramenta="car_details",
         deve=[num(cars.loc["c003"].consumption_kml), str(cars.loc["c003"].trunk_l)],
         gabarito=ficha("c003", "consumption_kml", "trunk_l")),
    dict(q=ECONOMICO, ferramenta="compare_cars|car_details", deve=[eco, num(cars.loc[eco].consumption_kml)],
         gabarito=f"{eco} is more economical (more km/l). " + "; ".join(ficha(i, "consumption_kml") for i in ["c081", "c082"])),
    dict(q="Entre o c109 e o c061, qual tem o maior porta-malas?", ferramenta="compare_cars|car_details",
         deve=[mala, str(cars.loc[mala].trunk_l)],
         gabarito=f"{mala} has the larger trunk. " + "; ".join(ficha(i, "trunk_l") for i in ["c109", "c061"])),
    dict(q="Quero um SUV até 80 mil para a família. Compare os dois mais novos e me diga qual você escolheria.",
         ferramenta="compare_cars", deve=[suvs.index[0]], body="SUV", max_price=80000,
         gabarito="The newest SUVs up to R$ 80,000: " + "; ".join(ficha(i, "year", "km", "price_brl", "consumption_kml", "trunk_l") for i in novos)
                  + ". Any choice is fine if the justification matches these numbers."),
    dict(q="Qual é a cor do c999?", ferramenta="car_details",
         deve=[r"n[ãa]o (existe|encontr|há|ha)|nenhum|inexist"], proibido=["preto", "branco", "cinza", "prata", "vermelh", "azul"],
         gabarito="There is no car c999 in the stock; the answer must say so and must not invent a color."),
]

for i, p in enumerate(PERGUNTAS, 1):
    print(f"{i}. {p['q']}\n   gabarito: {p['gabarito'][:150]}{'…' if len(p['gabarito']) > 150 else ''}\n")

1. Quero um SUV para a família, até uns 80 mil. O que você recomenda?
   gabarito: 22 SUVs cost up to R$ 80,000. The 5 newest: c082 Renault Duster year 2021, km 56500, price_brl 75000, transmission automatic; c081 Renault Duster year…

2. Procuro um hatch até 50 mil. Qual é o mais novo?
   gabarito: The newest hatch up to R$ 50,000 is c199 Renault Kwid year 2024, km 21200, price_brl 50000.

3. Em que cidade está o c109 e qual é a cor dele?
   gabarito: c109 Volkswagen Nivus city Natal, color vermelho

4. Qual é o consumo e o tamanho do porta-malas do c003?
   gabarito: c003 Chevrolet Tracker consumption_kml 11.4, trunk_l 393

5. Entre o c081 e o c082, qual é o mais econômico?
   gabarito: c081 is more economical (more km/l). c081 Renault Duster consumption_kml 10.9; c082 Renault Duster consumption_kml 10.4

6. Entre o c109 e o c061, qual tem o maior porta-malas?
   gabarito: c109 has the larger trunk. c109 Volkswagen Nivus trunk_l 415; c061 Jeep Renegade trunk_l 320

7. Quero um SUV at

In [11]:
def checagem(p, r):
    """Checagem automática. Devolve a lista de falhas (vazia = passou)."""
    resp, falhas = r["resposta"].lower(), []
    falhas += [f"faltou {d}" for d in p["deve"] if not re.search(d, resp)]
    falhas += [f"não devia ter {x}" for x in p.get("proibido", []) if re.search(x, resp)]
    for cid in set(re.findall(r"\bc\d{3}\b", resp)) & set(cars.index):        # os carros citados respeitam o filtro?
        if "body" in p and cars.loc[cid].body != p["body"]:
            falhas.append(f"{cid} é {cars.loc[cid].body}")
        if "max_price" in p and cars.loc[cid].price_brl > p["max_price"]:
            falhas.append(f"{cid} passa do orçamento")
    if not any(re.fullmatch(p["ferramenta"], f) for f in r["ferramentas"]):    # a trajetória
        falhas.append(f"não chamou {p['ferramenta']}")
    return falhas

class Veredito(BaseModel):
    correta: bool = Field(description="true if the answer agrees with the reference and adds no wrong fact")
    motivo: str = Field(description="one short sentence")

JUIZ = """You grade answers of a used-car assistant. You get the question, a reference with the true facts,
and the assistant's answer. The answer is correct if it agrees with the reference and every number, car_id
and comparison in it is true. Extra details that do not contradict the reference are fine.
Do not penalize an answer for being cut off at the end."""
juiz = modelo(max_tokens=200).with_structured_output(Veredito)

ARQUITETURAS = {"agente único": unico, "gerador + verificador": dupla, "supervisor": equipe}
linhas = []
for nome, app in ARQUITETURAS.items():
    print(f"── {nome} " + "─" * 50)
    for i, p in enumerate(PERGUNTAS, 1):
        r = await rodar(app, p["q"], ver=False)
        falhas = checagem(p, r)
        v = await juiz.ainvoke([SystemMessage(JUIZ), HumanMessage(
            f"QUESTION:\n{p['q']}\n\nREFERENCE:\n{p['gabarito']}\n\nANSWER:\n{r['resposta']}")])
        linhas.append(dict(arquitetura=nome, pergunta=i, automatica=not falhas, juiz=v.correta,
                           chamadas=r["chamadas"], segundos=r["segundos"], falhas="; ".join(falhas),
                           motivo=v.motivo, resposta=r["resposta"]))
        print(f"  {i}. auto {'✅' if not falhas else '❌'}  juiz {'✅' if v.correta else '❌'}  "
              f"{r['chamadas']:>2} chamadas {r['segundos']:>6} s   {'; '.join(falhas)[:60] or v.motivo[:60]}")

resultados = pd.DataFrame(linhas)
resultados.to_csv("resultados.csv", index=False)

── agente único ──────────────────────────────────────────────────


  1. auto ✅  juiz ✅   2 chamadas   21.7 s   A resposta está alinhada com a referência. Todos os carros l


  2. auto ✅  juiz ✅   2 chamadas    9.1 s   A resposta está correta e concorda com a referência. Todos o


  3. auto ✅  juiz ✅   2 chamadas    4.4 s   A resposta está correta e concorda com a referência. Todos o


  4. auto ✅  juiz ✅   2 chamadas    5.1 s   A resposta está correta, pois está alinhada com os dados da 


  5. auto ✅  juiz ❌   2 chamadas    8.2 s   A resposta está incorreta porque afirma que o c082 é mais ec


... (15 outputs left out)

  5. auto ✅  juiz ✅   8 chamadas   50.0 s   A resposta está correta e concorda com a referência. Ela afi


  6. auto ✅  juiz ✅   5 chamadas   42.9 s   A resposta está correta e concorda com a referência. Ela ide


  7. auto ✅  juiz ✅   8 chamadas   86.6 s   A resposta está correta. Ela compara os dois SUVs mais novos


  8. auto ❌  juiz ✅   5 chamadas   16.9 s   faltou n[ãa]o (existe|encontr|há|ha)|nenhum|inexist


In [12]:
tabela = resultados.groupby("arquitetura", sort=False).agg(
    acerto_automatico=("automatica", "sum"), acerto_juiz=("juiz", "sum"),
    chamadas_ao_modelo=("chamadas", "sum"), segundos=("segundos", "sum"))
tabela["acerto_automatico"] = tabela.acerto_automatico.astype(str) + "/8"
tabela["acerto_juiz"] = tabela.acerto_juiz.astype(str) + "/8"
tabela["segundos"] = tabela.segundos.round()
tabela

,acerto_automatico,acerto_juiz,chamadas_ao_modelo,segundos
arquitetura,,,,
agente único,8/8,7/8,17,101.0
gerador + verificador,8/8,8/8,27,259.0
supervisor,6/8,7/8,52,367.0


In [13]:
print("Onde a checagem automática e o juiz discordaram:\n")
for _, x in resultados[resultados.automatica != resultados.juiz].iterrows():
    print(f"[{x.arquitetura} · pergunta {x.pergunta}] auto {'✅' if x.automatica else '❌'} · juiz {'✅' if x.juiz else '❌'}")
    print(f"   resposta: {x.resposta.replace(chr(10), ' ')[:220]}")
    print(f"   juiz: {x.motivo}\n")

Onde a checagem automática e o juiz discordaram:

[agente único · pergunta 5] auto ✅ · juiz ❌
   resposta: O carro c082 é o mais econômico, com um consumo de 10,4 km/l, em comparação ao c081, que tem um consumo de 10,9 km/l.
   juiz: A resposta está incorreta porque afirma que o c082 é mais econômico, mas de acordo com a referência, o c081 é mais econômico (10.9 km/l vs 10.4 km/l).

[supervisor · pergunta 8] auto ❌ · juiz ✅
   resposta: O carro com o ID "c999" não foi encontrado no sistema. Por favor, verifique o ID e tente novamente. Se precisar, posso ajudar a buscar carros com base em outros critérios, como preço ou tipo de carro.
   juiz: A resposta está correta, pois informa que o carro com o ID 'c999' não foi encontrado, conforme indicado na referência. Não há inventação de cor ou informações incorretas.



### 🔍 O que aconteceu

| | Checagem automática | Juiz | Chamadas | Segundos |
|---|---|---|---|---|
| Agente único | 8/8 | 7/8 | 17 | 101 |
| Gerador + verificador | 8/8 | **8/8** | 27 | 259 |
| Supervisor | 6/8 | 7/8 | 52 | 367 |

- **O verificador foi o único 8/8 no juiz.** Ele consertou a pergunta 5, a do c081 contra o c082, ao custo de 1,6 vez as chamadas e 2,6 vezes o tempo do agente único.
- **O supervisor gastou 3 vezes as chamadas e quase 4 vezes o tempo, sem acertar mais.** Ele acertou a pergunta 5, mas errou a 3: disse que o c109 é **preto**, e o estoque diz vermelho. Com mais passagens entre agentes, há mais lugares para um dado se perder. Para saber onde, é preciso abrir a trajetória.
- **Os dois corretores erraram, em direções opostas.** Na pergunta 5 do agente único, a regra passou uma resposta errada, porque o c081 e o 10,9 estavam no texto, só que invertidos. O juiz pegou. Na pergunta 8 do supervisor, a regra reprovou uma resposta certa: ela esperava "não existe" ou "não encontr…", e a resposta disse "**não foi** encontrado". O juiz aprovou.
- **Oito perguntas é pouco.** Cada pergunta vale 12,5 pontos percentuais. Serve para comparar arquiteturas no mesmo problema e para achar os erros; não serve para publicar uma taxa de acerto.

Com o mesmo modelo e a temperatura 0, os acertos e as chamadas saíram iguais em duas execuções; só os segundos variaram um pouco. Numa GPU diferente, os segundos mudam, e uma ou outra resposta pode mudar.

---
# Parte 5 · As mesmas três no CrewAI

O CrewAI descreve a equipe de outro jeito. Em vez de nós e arestas, cada agente tem um **papel**, um **objetivo** e uma **história** (*backstory*), e cada **tarefa** diz o que fazer e quem faz. Aqui montamos as **mesmas três arquiteturas**, com o mesmo modelo e o mesmo `server.py`, e medimos com as mesmas 8 perguntas, a mesma checagem e o mesmo juiz da Parte 4.

| Arquitetura | LangGraph (Partes 1 a 3) | CrewAI |
|---|---|---|
| **Agente único** | o grafo `modelo ⇄ ferramentas` | um `Agent` com as três ferramentas e uma `Task` |
| **Gerador + verificador** | o código consulta o estoque; o verificador só aceita objeção com evidência e pode devolver a resposta | processo `sequential`: o vendedor responde e o verificador, que tem a `car_details`, reescreve a resposta |
| **Supervisor** | um roteador com saída estruturada escolhe o especialista; um redator escreve no fim | processo `hierarchical`: um agente gerente delega ao buscador e ao comparador e escreve a resposta |

O loop de cada agente fica dentro do `Agent`. Para ver o que aconteceu, o CrewAI emite **eventos**: cada chamada ao modelo e cada uso de ferramenta. Vamos ouvir esses eventos para contar as chamadas e montar a trajetória, como fizemos com as mensagens do LangGraph.

In [14]:
import logging
logging.getLogger("crewai").setLevel(logging.ERROR)                      # menos avisos na tela
from crewai import Agent, Task, Crew, Process, LLM
from crewai.events import crewai_event_bus
from crewai.events.types.llm_events import LLMCallCompletedEvent
from crewai.events.types.tool_usage_events import ToolUsageStartedEvent, ToolUsageFinishedEvent
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters

llm_crew = LLM(model=f"ollama_chat/{MODEL}", base_url=OLLAMA_URL, temperature=0,
               reasoning_effort="none", context_window_size=8192,          # o mesmo qwen3:8b, sem o modo de raciocínio
               max_tokens=800, frequency_penalty=0.3)                       # sem limite, o modelo pode entrar em loop repetindo frases

if "mcp_crew" in globals():
    mcp_crew.stop()                                                         # se a célula rodar de novo, fecha o servidor anterior
mcp_crew = MCPServerAdapter(StdioServerParameters(command=sys.executable, args=["server.py"],
                                                  env=SERVIDOR["estoque"]["env"]))   # sobe o server.py e faz tools/list
T = {f.name: f for f in mcp_crew.tools}
ESTOQUE = set(T)                                                            # as ferramentas do servidor, para a trajetória

RESPOSTA = "At most 4 sentences in Portuguese answering the request, citing the car_id of every car you mention"
VENDEDOR = dict(role="Used-car assistant in Natal",
                goal="Answer the customer's request with real cars from the stock, citing every car_id",
                backstory="You only trust the stock tools and never invent a car.")

def crew_unico():
    vendedor = Agent(**VENDEDOR, tools=list(T.values()), llm=llm_crew, max_iter=6)
    return Crew(agents=[vendedor], tasks=[Task(description="Customer request: {pedido}", expected_output=RESPOSTA, agent=vendedor)])

def crew_dupla():
    vendedor = Agent(**VENDEDOR, tools=list(T.values()), llm=llm_crew, max_iter=6)
    verificador = Agent(
        role="Fact checker", goal="Make sure every claim in the answer matches the stock data",
        backstory="You look up each car_id with car_details and fix any wrong number or ranking. More km/l means more economical.",
        tools=[T["car_details"]], llm=llm_crew, max_iter=6)
    responder = Task(description="Customer request: {pedido}", expected_output=RESPOSTA, agent=vendedor)
    conferir = Task(description="Customer request: {pedido}\nCheck the answer against the stock data and return the corrected final answer.",
                    expected_output=RESPOSTA, agent=verificador, context=[responder])   # a saída do vendedor vira contexto
    return Crew(agents=[vendedor, verificador], tasks=[responder, conferir], process=Process.sequential)

GERENTE = """You coordinate two specialists for a used-car assistant in Natal, Brazil. You have no stock tools yourself.
- buscador: finds cars by budget and body type, and gets the details of one car (tools: search_cars, car_details).
- comparador: compares 2 to 4 specific car_ids side by side (tool: compare_cars).
Delegate one precise task at a time, with car_ids and filters. Answer the user only with facts from their reports;
never invent a car or a car_id."""                                          # as mesmas instruções do supervisor da Parte 3

def crew_supervisor():
    buscador = Agent(role="buscador", goal="Find cars by budget and body type, and get the details of one car",
                     backstory="You are the search specialist. You use search_cars and car_details and report the facts with car_ids.",
                     tools=[T["search_cars"], T["car_details"]], llm=llm_crew, max_iter=6)
    comparador = Agent(role="comparador", goal="Compare 2 to 4 specific car_ids side by side",
                       backstory="You are the comparison specialist. You use compare_cars and report the facts with car_ids.",
                       tools=[T["compare_cars"]], llm=llm_crew, max_iter=6)
    gerente = Agent(role="supervisor", goal="Answer the customer's request by delegating to the right specialist",
                    backstory=GERENTE, llm=llm_crew, allow_delegation=True, max_iter=8)   # o gerente não tem ferramentas do estoque
    return Crew(agents=[buscador, comparador], tasks=[Task(description="Customer request: {pedido}", expected_output=RESPOSTA)],
                process=Process.hierarchical, manager_agent=gerente)

async def rodar_crew(montar, pergunta, ver=True):
    """Monta a crew, roda numa pergunta e devolve o mesmo dicionário do rodar() da Parte 1."""
    passos, t = [], time.time()
    with crewai_event_bus.scoped_handlers():                                # ouve os eventos só durante esta execução
        @crewai_event_bus.on(LLMCallCompletedEvent)
        def conta_chamada(fonte, ev):
            passos.append(dict(tipo="modelo", agente=ev.agent_role, t=time.time() - t))
        @crewai_event_bus.on(ToolUsageStartedEvent)
        def anota_delegacao(fonte, ev):                                     # o gerente passa uma tarefa a um colega
            if ev.tool_name not in ESTOQUE:
                passos.append(dict(tipo="delegou", agente=ev.agent_role, t=time.time() - t, args=ev.tool_args))
        @crewai_event_bus.on(ToolUsageFinishedEvent)
        def anota_ferramenta(fonte, ev):
            passos.append(dict(tipo="ferramenta", agente=ev.agent_role, t=time.time() - t,
                               nome=ev.tool_name, args=ev.tool_args, saida=str(ev.output)))
        saida = await montar().kickoff_async(inputs={"pedido": pergunta})
        crewai_event_bus.flush()                                            # espera os últimos eventos
    r = {"resposta": saida.raw,
         "ferramentas": [p["nome"] for p in passos if p["tipo"] == "ferramenta" and p["nome"] in ESTOQUE],
         "chamadas": sum(p["tipo"] == "modelo" for p in passos), "segundos": round(time.time() - t, 1), "passos": passos}
    if ver:
        print(f"👤 {pergunta}\n")
        mostrar_crew(r)
    return r

def mostrar_crew(r):
    """O trace a partir dos eventos: delegações, pedidos de ferramenta e resultados, com o tempo de cada um."""
    for p in r["passos"]:
        args = p.get("args") if isinstance(p.get("args"), dict) else {}
        quando = f"[{p['t']:5.1f} s]"
        if p["tipo"] == "delegou":                                          # delegar ou perguntar a um colega
            print(f"{quando} 🧭 supervisor → {args.get('coworker', '?')}: {str(args.get('task') or args.get('question', ''))[:140]}")
        elif p["tipo"] == "ferramenta" and p["nome"] in ESTOQUE:
            print(f"{quando} 🤖 {p['agente']:<10} pede → {p['nome']}({json.dumps(args, ensure_ascii=False)})")
            print(f"{'':10}📡 servidor   → {resumo(p['saida'])}")
        elif p["tipo"] == "ferramenta":                                     # o colega terminou e devolveu o relatório
            print(f"{quando} 📋 {args.get('coworker', '?'):<10} → {p['saida'].replace(chr(10), ' ')[:160]}…")
    print(f"\n💬 {r['resposta']}\n")
    print(f"⏱ {r['chamadas']} chamadas ao modelo · {r['segundos']} s")

print("crews prontas:", ", ".join(T))

crews prontas: search_cars, car_details, compare_cars


🔮 **Antes de rodar.** O pedido da Parte 1, agora para o supervisor do CrewAI. Compare com o trace da Parte 3: o gerente delega do mesmo jeito que o nosso supervisor roteava?

In [15]:
r = await rodar_crew(crew_supervisor, PEDIDO)

👤 Quero um SUV para a família, até uns 80 mil. O que você recomenda?

[ 17.5 s] 🧭 supervisor → buscador: Find SUVs within the budget of 80,000 reais and provide details for each car.
[ 21.5 s] 🤖 buscador   pede → search_cars({"max_price": 80000, "body": "SUV"})
          📡 servidor   → 22 encontrados; devolveu ['c082', 'c081', 'c109', 'c061', 'c003']
[ 24.3 s] 🤖 buscador   pede → car_details({"car_id": "c082"})
          📡 servidor   → c082 Duster SUV: 10.4 km/l, porta-malas 475 L, cinza, Macaíba
[ 26.1 s] 🤖 buscador   pede → car_details({"car_id": "c081"})
          📡 servidor   → c081 Duster SUV: 10.9 km/l, porta-malas 475 L, branco, Natal
[ 28.0 s] 🤖 buscador   pede → car_details({"car_id": "c109"})
          📡 servidor   → c109 Nivus SUV: 12.5 km/l, porta-malas 415 L, vermelho, Natal
[ 29.9 s] 🤖 buscador   pede → car_details({"car_id": "c061"})
          📡 servidor   → c061 Renegade SUV: 10.0 km/l, porta-malas 320 L, cinza, Mossoró
[ 31.8 s] 🤖 buscador   pede → car_details({"car_id

### 🔍 O que aconteceu

- O gerente não tem ferramenta do estoque. Para delegar, ele chama uma ferramenta que o próprio CrewAI cria, a *Delegate work to coworker*: **no CrewAI, delegar também é uma chamada de ferramenta**. No trace, é a linha 🧭.
- O buscador recebeu "encontrar SUVs até 80 mil e trazer os detalhes de cada um". Chamou a `search_cars` e depois a `car_details` para cada um dos cinco carros. Depois, o comparador comparou os cinco com a `compare_cars`.
- A resposta lista três carros com os números certos.
- Olhe os tempos. A última ferramenta do buscador terminou em 32 s, e o relatório dele chegou em 73 s: foram **41 s escrevendo uma tabela de cinco carros** para o gerente ler. O comparador levou mais 32 s no relatório dele. A maior parte do tempo foi gasta nas passagens entre agentes, não nas ferramentas.
- Foram **12 chamadas e 137 s**. Na Parte 4, o supervisor do LangGraph respondeu a mesma pergunta com 8 chamadas.

Um detalhe da preparação. Na primeira versão, usamos o gerente padrão do CrewAI (`manager_llm`), que só conhece o papel de cada agente. Ele mandou a busca para o comparador, que não tem a `search_cars`, e a resposta final citou carros que não existem, com `car_id` 12345. Por isso o gerente aqui é um `manager_agent` com as mesmas instruções do supervisor da Parte 3. No CrewAI, como no LangGraph, quem coordena precisa saber o que cada especialista faz.

Agora a medição: as três crews nas 8 perguntas da Parte 4, com a mesma checagem automática e o mesmo juiz. As linhas entram na mesma lista `linhas`, com o nome do framework, para a tabela final comparar as seis.

In [16]:
CREWS = {"agente único": crew_unico, "gerador + verificador": crew_dupla, "supervisor": crew_supervisor}
for linha in linhas:
    linha.setdefault("framework", "LangGraph")                              # as linhas da Parte 4
linhas = [l for l in linhas if l["framework"] == "LangGraph"]               # se a célula rodar de novo, não duplica
for nome, montar in CREWS.items():
    print(f"── {nome} · CrewAI " + "─" * 40)
    for i, p in enumerate(PERGUNTAS, 1):
        r = await rodar_crew(montar, p["q"], ver=False)
        falhas = checagem(p, r)
        v = await juiz.ainvoke([SystemMessage(JUIZ), HumanMessage(
            f"QUESTION:\n{p['q']}\n\nREFERENCE:\n{p['gabarito']}\n\nANSWER:\n{r['resposta']}")])
        linhas.append(dict(framework="CrewAI", arquitetura=nome, pergunta=i, automatica=not falhas, juiz=v.correta,
                           chamadas=r["chamadas"], segundos=r["segundos"], falhas="; ".join(falhas),
                           motivo=v.motivo, resposta=r["resposta"]))
        print(f"  {i}. auto {'✅' if not falhas else '❌'}  juiz {'✅' if v.correta else '❌'}  "
              f"{r['chamadas']:>2} chamadas {r['segundos']:>6} s   {'; '.join(falhas)[:60] or v.motivo[:60]}")
mcp_crew.stop()                                                             # fecha o servidor do CrewAI

resultados = pd.DataFrame(linhas)
resultados.to_csv("resultados.csv", index=False)

── agente único · CrewAI ────────────────────────────────────────


  1. auto ✅  juiz ✅   3 chamadas   25.4 s   A resposta está correta, pois recomenda SUVs dentro do orçam


  2. auto ✅  juiz ✅   2 chamadas   14.3 s   A resposta está correta. Ela menciona o Kwid (car_id: c199) 


  3. auto ✅  juiz ✅   2 chamadas    8.6 s   A resposta está correta e concorda com a referência. A cidad


  4. auto ✅  juiz ✅   2 chamadas    9.6 s   A resposta está correta e concorda com a referência. Todos o


  5. auto ✅  juiz ✅   2 chamadas   12.2 s   A resposta está correta. Ela afirma que o c081 é mais econôm


... (15 outputs left out)

  5. auto ✅  juiz ✅   4 chamadas   22.6 s   A resposta está correta e concorda com a referência. Ela ide


  6. auto ✅  juiz ✅   4 chamadas   22.6 s   A resposta está correta, pois afirma que o c109 tem o maior 


  7. auto ✅  juiz ✅   9 chamadas   77.8 s   A resposta escolhe o Renault Duster (c082), que está dentro 


  8. auto ❌  juiz ✅   4 chamadas   17.9 s   faltou n[ãa]o (existe|encontr|há|ha)|nenhum|inexist


In [17]:
seis = resultados.groupby(["arquitetura", "framework"], sort=False).agg(
    acerto_automatico=("automatica", "sum"), acerto_juiz=("juiz", "sum"),
    chamadas_ao_modelo=("chamadas", "sum"), segundos=("segundos", "sum"))
seis["acerto_automatico"] = seis.acerto_automatico.astype(str) + "/8"
seis["acerto_juiz"] = seis.acerto_juiz.astype(str) + "/8"
seis["segundos"] = seis.segundos.round()
seis

,,acerto_automatico,acerto_juiz,chamadas_ao_modelo,segundos
arquitetura,framework,,,,
agente único,LangGraph,8/8,7/8,17,101.0
gerador + verificador,LangGraph,8/8,8/8,27,259.0
supervisor,LangGraph,6/8,7/8,52,367.0
agente único,CrewAI,8/8,8/8,18,114.0
gerador + verificador,CrewAI,8/8,8/8,30,202.0
supervisor,CrewAI,7/8,8/8,46,352.0


In [18]:
print("CrewAI: onde a checagem automática e o juiz discordaram, e as respostas que o juiz reprovou:\n")
crew = resultados[resultados.framework == "CrewAI"]
for _, x in crew[(crew.automatica != crew.juiz) | ~crew.juiz].iterrows():
    print(f"[{x.arquitetura} · pergunta {x.pergunta}] auto {'✅' if x.automatica else '❌'} · juiz {'✅' if x.juiz else '❌'}")
    print(f"   resposta: {x.resposta.replace(chr(10), ' ')[:220]}")
    print(f"   regra: {x.falhas or '—'}")
    print(f"   juiz: {x.motivo}\n")

CrewAI: onde a checagem automática e o juiz discordaram, e as respostas que o juiz reprovou:

[supervisor · pergunta 8] auto ❌ · juiz ✅
   resposta: O carro com o ID c999 não está disponível em nosso banco de dados. Por favor, verifique o ID do carro e tente novamente.
   regra: faltou n[ãa]o (existe|encontr|há|ha)|nenhum|inexist
   juiz: A resposta está correta, pois informa que o carro com o ID c999 não está disponível em seu banco de dados, o que está de acordo com a referência.



### 🔍 O que aconteceu

| | Framework | Checagem automática | Juiz | Chamadas | Segundos |
|---|---|---|---|---|---|
| Agente único | LangGraph | 8/8 | 7/8 | 17 | 101 |
| | CrewAI | 8/8 | **8/8** | 18 | 114 |
| Gerador + verificador | LangGraph | 8/8 | 8/8 | 27 | 259 |
| | CrewAI | 8/8 | 8/8 | 30 | 202 |
| Supervisor | LangGraph | 6/8 | 7/8 | 52 | 367 |
| | CrewAI | 7/8 | 8/8 | 46 | 352 |

- **O agente único do CrewAI acertou a pergunta 5, que o do LangGraph errou.** O modelo, as ferramentas e a pergunta são os mesmos. O que muda é o texto que o modelo recebe. O CrewAI monta o prompt do agente com o papel, o objetivo e a história (*"You are Used-car assistant in Natal..."*) e acrescenta o formato esperado da tarefa ("no máximo 4 frases"). **O framework escreve parte do prompt.** Por isso, compare arquiteturas dentro do mesmo framework, e não um framework contra o outro.
- **Dentro do CrewAI, a conclusão da Parte 4 se repete no supervisor.** Foram 46 chamadas, 2,6 vezes as do agente único, e 3 vezes o tempo, para o mesmo 8/8 no juiz.
- **O verificador empatou no placar e mesmo assim fez diferença.** Na pergunta 5, o agente único escreveu que o c082 "tem mais quilometragem". Não tem: são 56.500 km, contra 67.700 do c081. O juiz aprovou, porque o gabarito dessa pergunta só fala do consumo. O verificador consultou os carros e tirou a frase. Um 8/8 quer dizer "nenhum erro que o gabarito cobre", e não "nenhum erro".
- **O verificador do CrewAI levou menos tempo que o nosso**, 202 s contra 259 s, mesmo com mais chamadas. O nosso, da Parte 2, liga o raciocínio do qwen3 para julgar, e cada chamada fica mais longa.
- A única discordância entre os corretores é do mesmo tipo da Parte 4. Na pergunta 8, o supervisor escreveu "não está disponível", e a regra esperava "não existe" ou "não encontrado". O juiz aprovou.

Os acertos e as chamadas saíram iguais em duas execuções; só os segundos variaram, como na Parte 4. Os segundos do LangGraph na tabela são os da Parte 4.

| | LangGraph | CrewAI |
|---|---|---|
| **Você descreve** | o fluxo: nós, arestas e estado | a equipe: papéis, objetivos e tarefas |
| **O loop de cada agente** | à vista, no grafo | escondido dentro do `Agent` |
| **O supervisor** | um nó que roteia, com saída estruturada | um gerente que delega por uma ferramenta |
| **Verificação ancorada em evidência** | escrevemos em código (`tem_evidencia`) | fica nas instruções do agente |
| **Para ver o que aconteceu** | as mensagens do estado | os eventos (`crewai_event_bus`) |
| **Bom para** | controlar cada passo e medir | montar uma equipe rápido |

---
# Resumo

```
Parte 1 · agente único            modelo ⇄ ferramentas
Parte 2 · gerador + verificador   gerador ⇄ ferramentas → verificador (código consulta, modelo julga) → volta ou fim
Parte 3 · supervisor              supervisor → buscador / comparador → … → redator
Parte 5 · as mesmas três no CrewAI: Agent + Task, processo sequential ou hierarchical
                                         todos usando o mesmo server.py da aula 11, por MCP
```

**O que a medição mostrou, neste problema:**

- O **verificador** do LangGraph acertou mais (8/8) e custou 1,6 vez as chamadas. Ele vale a pena porque existe um critério claro para conferir, o estoque, e porque a consulta é feita pelo código.
- O **supervisor** foi o mais caro nos dois frameworks: de 2,6 a 3,6 vezes as chamadas e o tempo do agente único, sem acertar mais. Para três ferramentas e perguntas curtas, um agente só dava conta do contexto.
- **O framework escreve parte do prompt.** O mesmo agente único acertou uma pergunta a mais no CrewAI. Compare arquiteturas no mesmo framework.
- A **checagem automática** e o **juiz** erraram em direções opostas, e um 8/8 ainda deixou passar uma frase errada que o gabarito não cobria. Use os dois e leia as respostas.

É a conclusão do artigo da Anthropic de outubro de 2026: comece com um agente, construa a avaliação primeiro e só divida quando os números pedirem.

### Para o trabalho

O trabalho que começa na quarta pede exatamente isto: escolher uma arquitetura, montar com um framework (LangGraph, CrewAI ou outro) e **mostrar com medida** por que ela vale o que custa. Reaproveite o método da Parte 4: perguntas com gabarito tirado dos dados, checagem automática da resposta e da trajetória, e um juiz com referência.